# Setup and data acquisition

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yazanjer/An_Explainable_AI_Education/blob/main/notebooks/00_setup_and_data.ipynb)

**Answers:** Editor comment 3 (sample description)
**Estimated runtime:** 10-40 min (one-off download) · **Hardware:** CPU
**Quick mode:** set `QUICK_MODE = True` in the setup cell for a fast smoke test.

Downloads `SPSS_STU_QQQ.zip` from the OECD, verifies its checksum, converts SPSS to
parquet **once**, and caches to Drive. Every later notebook reads the cached parquet
and fails with an actionable message if it is missing.

PISA data is **not redistributed** with this repository. See the data-availability
statement in the README.

---


In [ ]:
# --- Environment setup -------------------------------------------------
# Detects Colab, mounts Drive only when in Colab, installs pinned deps.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
QUICK_MODE = True   # set False for the full budget

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT = Path("/content/drive/MyDrive/An_Explainable_AI_Education")
    PROJECT.mkdir(parents=True, exist_ok=True)
    if not (PROJECT / "src").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/yazanjer/An_Explainable_AI_Education.git", str(PROJECT)],
                       check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT / "requirements.txt")], check=False)
else:
    PROJECT = Path(os.environ.get("VLPSO_PROJECT_ROOT", Path.cwd().parent))

os.environ["VLPSO_PROJECT_ROOT"] = str(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

from vlpso_xai.config import load_config, set_global_seeds, environment_report
cfg = load_config("quick" if QUICK_MODE else "default")
set_global_seeds(cfg.seed)
cfg.paths.mkdirs()
print("project root:", cfg.paths.root)
print("config:", cfg.config_path.name, "| hash:", cfg.hash()[:12])


In [ ]:
# --- Download and cache (skips if already present) ---------------------
import hashlib, subprocess, sys
from pathlib import Path

RAW = cfg.paths.data_raw; RAW.mkdir(parents=True, exist_ok=True)
SAV = RAW / cfg.section("data", "source", "student_sav")
URL = cfg.section("data", "source", "student_zip_url")

if not SAV.exists():
    zp = RAW / "SPSS_STU_QQQ.zip"
    if not zp.exists():
        print("Downloading ~500 MB from the OECD ...")
        subprocess.run(["curl", "-L", "-o", str(zp), URL], check=True)
    print("sha256:", hashlib.sha256(zp.read_bytes()).hexdigest())
    subprocess.run(["unzip", "-o", "-q", str(zp), "-d", str(RAW)], check=True)
else:
    print("Found cached", SAV.name)

assert SAV.exists(), (
    f"{SAV} is missing. Download the PISA 2018 student questionnaire from\n"
    f"{cfg.section('data','source','landing_page')} and place the .sav here."
)

In [ ]:
# --- Extract the analytic countries, cache as parquet ------------------
import numpy as np, pandas as pd, pyreadstat

PROC = cfg.paths.data_processed; PROC.mkdir(parents=True, exist_ok=True)
OUT = PROC / "analytic.parquet"

if not OUT.exists():
    _, meta = pyreadstat.read_sav(str(SAV), metadataonly=True)
    countries = [cfg.section("data", "primary_country")] + list(
        cfg.section("data", "external_countries"))
    cnt, _ = pyreadstat.read_sav(str(SAV), usecols=["CNT"])
    keep = cnt["CNT"].isin(countries).to_numpy()
    print({c: int((cnt["CNT"] == c).sum()) for c in countries})

    # Full-width missingness, needed for the exclusion rule (editor comment 3)
    miss = np.zeros(len(cnt), dtype=np.int32)
    for i in range(0, len(meta.column_names), 250):
        d, _ = pyreadstat.read_sav(str(SAV), usecols=meta.column_names[i:i+250])
        miss += d.isna().sum(axis=1).to_numpy(dtype=np.int32); del d
    df, _ = pyreadstat.read_sav(str(SAV))
    df["n_missing_allcols"] = miss
    df = df[keep].reset_index(drop=True)
    df.to_parquet(OUT, index=False)
print("cached:", OUT)

In [ ]:
# --- Sample flow (editor comment 3) ------------------------------------
import pandas as pd
from vlpso_xai.data.outcome import build_pv_categories, category_instability

df = pd.read_parquet(OUT)
thr = cfg.section("data", "missing_row_threshold")
rows = [dict(step="raw country subset", n=len(df), schools=df.CNTSCHID.nunique())]
for t in [thr] + list(cfg.section("data", "missing_row_threshold_sensitivity")):
    k = df[df.n_missing_allcols <= t]
    rows.append(dict(step=f"missingness <= {t}", n=len(k), schools=k.CNTSCHID.nunique()))
flow = pd.DataFrame(rows)
flow.to_csv(cfg.paths.results / "sample" / "sample_flow.csv", index=False)
display(flow)

prim = df[df.n_missing_allcols <= thr].reset_index(drop=True)
cats = build_pv_categories(prim)
print(category_instability(cats, prim.W_FSTUWT.to_numpy()))